# Neutrino mass: from a measurement to the minimal model that explains it

The Standard Model has no neutrino mass. Oscillation experiments measure two nonzero
mass splittings. This notebook climbs the **ladder** between those two facts: a dimensional
estimate, then the SM's own field content, then an effective operator, then a tree-level model,
and finally a fit to data.

The pedagogy is **compute step by step, then develop judgment**. Nothing below is typed in from
the answer. The missing mass term is *derived* by enumerating invariants. The Weinberg operator's
effect is *read off* by putting the Higgs in its vacuum. The need for a second $\nu_R$ is *proved*
as a matrix rank, and only then *checked* against the data.

Each section runs four moves: **set up** (look at the raw object), **collect** (let the structure
emerge), **recognise** (name what appeared and why it had to), and **check** (by an independent
route, not by re-running the same algebra). Before the cell that settles each question, a
blockquote asks you to commit to an expectation.

**The one question this notebook is really about.** The SM predicts massless neutrinos, and
oscillations say otherwise. What is the *least* you must add to the SM, and how much of the
answer can you know **before building a model**?

*Housekeeping.* Data, sources and status live in [`anomaly.yaml`](anomaly.yaml). They are loaded
below and never re-typed as literals. Decisions are logged in [`decisions.md`](decisions.md).
The notebook ships solved, so that `pytest --nbmake` can run it top to bottom in CI. To work
through it as an exercise, make your own copy and leave `checkpoints_def.py` / `solutions/`
alone until you want a hint. Claims are tagged *(computed)* when a cell here produces them, and
*(cited)* when they come from the literature.

In [1]:
import math
import sys
from pathlib import Path

import numpy as np
import sympy as sp
from IPython.display import Markdown, Math, display

# nbclient/nbmake set the kernel's cwd to this notebook's own directory; make sure it (and the
# repo root two levels up, for the `anomalies`/`checkpoints`/`fit`/`feynlag_anomalies` packages)
# are importable regardless of how the kernel was launched.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parents[1]
for p in (NB_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from feynlag_anomalies.latex import latex
from feynlag_anomalies.registry import load as load_anomaly
from anomalies.neutrino_mass import checkpoints_def as cd

# text/latex only, and feynlag names in physics notation (nuL -> \nu_L, Gp -> G^+)
sp.init_printing(use_latex="mathjax", latex_printer=latex)


def ok(msg):
    print("✓", msg)


def trace(msg, obj=None):
    # One line of commentary, optionally followed by a sympy object.  The cells
    # that assert stay quiet; the "peek" cells do the talking through here.
    print(msg)
    if obj is not None:
        display(obj)


GEV_TO_EV = 1e9

anomaly = load_anomaly("neutrino_mass")
obs = {o.name: o for o in anomaly.observables}
print(anomaly.title, "--", anomaly.maturity)

Neutrino mass and oscillations -- A3


---
## Prelude — what oscillations measure

A neutrino produced with flavour $\alpha$ is a superposition of mass eigenstates. If those have
different masses, they pick up different phases while travelling, and the flavour content changes
with distance. In the two-flavour approximation *(cited: the standard oscillation formula, e.g.
the PDG review "Neutrino masses, mixing, and oscillations")*:

$$P(\nu_\alpha \to \nu_\beta) = \sin^2 2\theta \; \sin^2\!\left(\frac{\Delta m^2 L}{4E}\right),
\qquad \Delta m^2 = m_2^2 - m_1^2 ,$$

in natural units, for baseline $L$ and energy $E$. The amplitude measures a mixing angle, one
entry of the PMNS matrix with its three angles $\theta_{12}, \theta_{13}, \theta_{23}$ and a phase
$\delta_{CP}$. The oscillation frequency measures a splitting.

> **Before running the next cell.** Suppose every neutrino were massless. What would $P$ be for
> $\alpha \neq \beta$, at any $L$ and $E$? And what can oscillations *never* tell you, however
> precise they become: a mass, or only a difference of squared masses?

In [2]:
# ---- MOVE 1: set up.  The measured parameters, straight from anomaly.yaml. ----
rows = ["| observable | value | 1σ | units |", "|---|---|---|---|"]
rows += [f"| {o.name} | {o.value:.4g} | {o.stat_uncertainty:.2g} | {o.units} |" for o in anomaly.observables]
source = anomaly.sources[0]
display(Markdown("\n".join(rows) + f"\n\nSource: {source.identifier}, consulted {source.consulted_on}."))

dm2 = [obs[n] for n in ("Delta m^2_21 (solar)", "Delta m^2_31 (atmospheric, normal ordering)")]
assert all(o.value / o.stat_uncertainty > 5 for o in dm2)
ok("both splittings are more than 5 sigma from zero -- at least two neutrinos are massive")

| observable | value | 1σ | units |
|---|---|---|---|
| Delta m^2_21 (solar) | 7.49e-05 | 1.9e-06 | eV^2 |
| Delta m^2_31 (atmospheric, normal ordering) | 0.002513 | 2e-05 | eV^2 |
| theta_12 | 33.68 | 0.71 | degrees |
| theta_13 | 8.56 | 0.11 | degrees |
| theta_23 | 43.3 | 0.9 | degrees |
| delta_CP | 212 | 34 | degrees |

Source: arXiv:2410.05380v2 (NuFIT 6.0; JHEP 12 (2024) 216; INSPIRE 2838825), Table 1, IC24 with SK atmospheric data, Normal Ordering, consulted 2026-09-25.

✓ both splittings are more than 5 sigma from zero -- at least two neutrinos are massive


**Read it.** With massless neutrinos, $\Delta m^2 = 0$ and $P = 0$: no flavour change, ever.
Two independent splittings, solar $\Delta m^2_{21}$ and atmospheric $\Delta m^2_{31}$, are both
far from zero *(computed)*, so at least two neutrinos are massive. Oscillations fix only mass
*differences*, never the absolute scale; that gap stays open to the end of the notebook.

The SM has no neutrino mass term, so it predicts both splittings to be exactly zero. The rest of
this notebook asks *why* it has none, and what is the least we must add.

---
## 0. How small is small? A dimensional estimate

Neutrino masses are tiny compared with every other fermion's. Before any model, ask what
dimensional analysis allows.

> **Before running the next cell.** Suppose the mass comes from a Yukawa coupling $y$ to the
> Higgs, with electroweak scale $v$, and is suppressed by some heavy scale $M$. Which combination
> of $y$, $v$ and $M$ has units of mass and gets *smaller* as $M$ grows? Evaluate it with an
> $O(1)$ Yukawa, $v = 246\,\text{GeV}$, and a GUT-ish $M \sim 10^{14}\,\text{GeV}$.

In [3]:
# ---- MOVE 1 + 2: the estimate, checked against the exercise's checkpoint. ----
from anomalies.neutrino_mass.solutions.step_0 import m_nu_estimate

answer_0 = m_nu_estimate(cd._P0_Y, cd._P0_V, cd._P0_M)
assert cd.check_0(answer_0)

[step_0] correct -- got 6.0516e-10


The answer is

$$m_\nu \sim \frac{y^2 v^2}{M} :$$

two powers of $v$, one Higgs per neutrino leg, divided by one power of the heavy scale. How does
it compare with the data? Every neutrino mass is at least as large as the scale set by the
splittings, and $\sqrt{\Delta m^2_{31}}$ is the natural target.

In [4]:
# ---- MOVE 3 + 4: recognise the scale, then check it against the data. ----
dm2_31 = obs["Delta m^2_31 (atmospheric, normal ordering)"]
m_atm_eV = math.sqrt(dm2_31.value)
ratio_0 = answer_0 * GEV_TO_EV / m_atm_eV
print(f"  step-0 estimate     m_nu ~ {answer_0 * GEV_TO_EV:.3g} eV")
print(f"  sqrt(Delta m^2_31)       = {m_atm_eV:.3g} eV   (computed from NuFIT 6.0)")
print(f"  ratio                    = {ratio_0:.3g}")
assert 1 / 30 < ratio_0 < 30
ok("an O(1) Yukawa and M ~ 1e14 GeV land within about an order of magnitude of the data")

  step-0 estimate     m_nu ~ 0.605 eV
  sqrt(Delta m^2_31)       = 0.0501 eV   (computed from NuFIT 6.0)
  ratio                    = 12.1
✓ an O(1) Yukawa and M ~ 1e14 GeV land within about an order of magnitude of the data


**The judgment to keep:** a sub-eV mass does not need a tiny coupling. It can equally be an
ordinary coupling divided by an enormous scale. Neutrino masses may be the low-energy trace of
physics far above the electroweak scale. Section 4 shows that "the scale" is really a
*combination* of mass and coupling.

---
## 1. The SM has no room for a neutrino mass

The *minimal* SM lepton content is a lepton doublet `Ll`, a charged-lepton singlet `eR` and the
Higgs doublet `H`, with **no** right-handed neutrino. Start from their quantum numbers, read from
the field declarations rather than typed in.

> **Before running the next cells.** Can you write a renormalizable (dimension $\le 4$),
> gauge-invariant neutrino mass term with only these fields? A mass term pairs two fermions. List
> the candidate pairs, with or without one Higgs, and decide which can have total hypercharge
> zero.

In [5]:
# ---- MOVE 1: set up.  The fields and their charges, from the declarations. ----
from anomalies.neutrino_mass.solutions.step_1 import build_sm_lepton_fields

Ll, eR, H, groups = build_sm_lepton_fields()
SU2L, U1Y = groups
rows = ["| field | SU(2)$_L$ | $Y$ | components |", "|---|---|---|---|"]
for f in (Ll, eR, H):
    rows.append(f"| `{f.name}` | {f.reps.get(SU2L, 1)} | {f.reps.get(U1Y, 0)} | "
                f"{', '.join(f'${latex(c)}$' for c in f.components)} |")
display(Markdown("\n".join(rows)))

| field | SU(2)$_L$ | $Y$ | components |
|---|---|---|---|
| `Ll` | 2 | -1/2 | $\nu_L$, $e_L$ |
| `eR` | 1 | -1 | $e_R$ |
| `H` | 2 | 1/2 | $G^+$, $H^0$ |

In [6]:
# ---- MOVE 2: collect.  A hypercharge ledger for the candidate terms. ------
# A bar (or the conjugate Higgs H~) flips the sign of Y.  Each candidate is a
# list of (field, sign); the total is summed from the declared reps.
Y = lambda f: f.reps.get(U1Y, 0)
candidates = {
    r"\bar L\, H\, e_R": ([(Ll, -1), (H, +1), (eR, +1)], "charged-lepton Yukawa"),
    r"\bar L\, \tilde H": ([(Ll, -1), (H, -1)], "needs a partner fermion: a $\\nu_R$"),
    r"L\, L": ([(Ll, +1), (Ll, +1)], "Majorana, no Higgs"),
    r"L\, H\, L\, H": ([(Ll, +1), (H, +1), (Ll, +1), (H, +1)], "two Higgs legs: dimension 5"),
}
ledger = {name: sum(s * Y(f) for f, s in legs) for name, (legs, _) in candidates.items()}
rows = ["| candidate | $\\sum Y$ | note |", "|---|---|---|"]
rows += [f"| ${name}$ | {ledger[name]} | {note} |" for name, (_, note) in candidates.items()]
display(Markdown("\n".join(rows)))

| candidate | $\sum Y$ | note |
|---|---|---|
| $\bar L\, H\, e_R$ | 0 | charged-lepton Yukawa |
| $\bar L\, \tilde H$ | 0 | needs a partner fermion: a $\nu_R$ |
| $L\, L$ | -1 | Majorana, no Higgs |
| $L\, H\, L\, H$ | 0 | two Higgs legs: dimension 5 |

**Recognise.** Only two candidates are neutral:

- $\bar L H e_R$ has $\sum Y = 0$. This is the charged-lepton mass.
- $\bar L \tilde H$ also has $\sum Y = 0$ on its own. To be a *mass term* it still needs a
  fermion partner with $Y = 0$, a $\nu_R$, and the SM has none.
- $LL$ has $Y = -1$. No renormalizable term can fix that without a new field carrying $Y = +1$.
- $LHLH$ is neutral, but it has two Higgs legs, so it is dimension 5, not 4.

So the ledger predicts exactly **one** dimension-$\le 4$ lepton Yukawa. Now check that with an
independent tool. `feynlag.suggest.suggest_yukawa` enumerates every invariant contraction and
re-verifies each one (gauge, discrete, hermiticity, mass dimension).

In [7]:
# ---- MOVE 4: check.  The library's enumerator, independently of the ledger. ----
from feynlag import suggest_yukawa

terms_dim4 = suggest_yukawa([Ll, eR], [H], list(groups), max_dim=4)
for t in terms_dim4:
    display(Math(rf"\text{{{t.label} (dim {t.dim})}}:\quad {latex(t.expr)}"))

answer_1 = len(terms_dim4)
assert cd.check_1(answer_1)
assert answer_1 == sum(1 for n, y in ledger.items() if y == 0 and n.count("L") == 1 and "e_R" in n)
ok("suggest_yukawa and the hypercharge ledger agree: one dim <= 4 term, and it is not a neutrino mass")

<IPython.core.display.Math object>

[step_1] correct -- got 1
✓ suggest_yukawa and the hypercharge ledger agree: one dim <= 4 term, and it is not a neutrino mass


Worth looking at the enumerator's result directly before trusting the count. It is a
`SuggestedTerm` whose `.expr` is the full operator, written in doublet components:

In [8]:
# ---- peek: the one surviving term, as an object. ----
terms_dim4[0].expr

                                                                       __      ↪
Gp⋅Bilinear(nuLbar[i], PR, eR[j]) + H₀⋅Bilinear(eLbar[i], PR, eR[j]) + Gp⋅Bili ↪

↪                              __                              
↪ near(eRbar[j], PL, nuL[i]) + H₀⋅Bilinear(eRbar[j], PL, eL[i])

**The judgment to keep:** two protections block a neutrino mass at dimension $\le 4$ *(computed)*;
see [`docs/protections.md`](../../docs/protections.md).

- **Field content.** There is no $\nu_R$ to pair with $\nu_L$.
- **Accidental symmetry.** Assign lepton number $L = 1$ to `Ll` and `eR`, and $L = 0$ to `H`.
  Every renormalizable SM term then conserves $L$, without anyone imposing it. A Majorana mass
  $\nu_L\nu_L$ carries $\Delta L = 2$, so it can appear only once that accidental symmetry is
  broken.

Either protection can be relaxed: add a field, or go beyond dimension 4. The next section does
the second.

---
## 2. One operator at dimension 5

The ledger already found a neutral candidate, $LHLH$, with two Higgs legs. It is dimension 5,
so it is suppressed by one power of a heavy scale $\Lambda$.

> **Before running the next cell.** Raise `max_dim` to 5. How many *new* invariants do you expect?
> What is their lepton number? And once the Higgs gets its vacuum value, which component fields
> can such a term still contain?

In [9]:
# ---- MOVE 1 + 2: set up and collect.  The enumeration at dimension 5. ----
terms_dim5 = suggest_yukawa([Ll, eR], [H], list(groups), max_dim=5)
for t in terms_dim5:
    display(Math(rf"\text{{{t.label} (dim {t.dim})}}:\quad {latex(t.expr)}"))

answer_2 = any("Weinberg" in t.label for t in terms_dim5)
assert cd.check_2(answer_2)
assert len(terms_dim5) - len(terms_dim4) == 1
ok("exactly one new invariant at dimension 5")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[step_2] correct -- got True
✓ exactly one new invariant at dimension 5


**Recognise.** The new invariant is the **Weinberg operator** $(LH)(LH)/\Lambda$ *(cited: S.
Weinberg, Phys. Rev. Lett. 43 (1979) 1566)*. It has two lepton doublets and two Higgs doublets,
carries $\Delta L = 2$, and is the *unique* dimension-5 operator built from SM fields
*(computed above)*.

### 2.1 The Weinberg operator in the vacuum

The expression above is written in doublet components ($G^+$, $H^0$, $e_L$, $\nu_L$). To see what
it does at low energy, put the Higgs in its vacuum. Following the feynlag-models conventions,
$H = \big(G^+,\ (v + h + iG^0)/\sqrt2\big)$, so the vacuum is $G^+ \to 0$, $H^0 \to v/\sqrt2$.
Rather than substituting into the whole sum at once, walk it term by term:

In [10]:
# ---- peek: the vacuum substitution, one term at a time. ----
Gp, H0 = H.components
v = sp.Symbol("v", positive=True)
vacuum = {Gp: 0, H0: v / sp.sqrt(2)}
weinberg = next(t for t in terms_dim5 if "Weinberg" in t.label)

survivors = []
for term in weinberg.expr.args:
    image = sp.expand(term.subs(vacuum))
    tag = "survives" if image != 0 else "vanishes"
    trace(f"  {tag}:", sp.Eq(term, image, evaluate=False) if image != 0 else term)
    if image != 0:
        survivors.append(image)

  vanishes:


  2                                     
Gp ⋅MajoranaBilinear(eL[i], C⋅PL, eL[j])

  survives:


                                              2                                ↪
  2                                          v ⋅MajoranaBilinear(nuL[i], C⋅PL, ↪
H₀ ⋅MajoranaBilinear(nuL[i], C⋅PL, nuL[j]) = ───────────────────────────────── ↪
                                                                 2             ↪

↪         
↪  nuL[j])
↪ ────────
↪         

  vanishes:


...

-Gp⋅H₀⋅MajoranaBilinear(eL[j], C⋅PL, nuL[i])

  vanishes:


 __ __                                            
-Gp⋅H₀⋅MajoranaBilinear(eLbar[i], C⋅PR, nuLbar[j])

  vanishes:


 __ __                                            
-Gp⋅H₀⋅MajoranaBilinear(eLbar[j], C⋅PR, nuLbar[i])

In [11]:
# ---- MOVE 3 + 4: recognise the survivors, then check they are a nu_L mass. ----
at_vacuum = sp.expand(weinberg.expr.subs(vacuum))
assert sp.simplify(at_vacuum - sum(survivors)) == 0
legs = {str(i.base.label) for i in at_vacuum.atoms(sp.Indexed)}
assert legs <= {"nuL", "nuLbar"}, legs
display(Math(r"(LH)(LH)\big|_{\langle H \rangle} = " + latex(at_vacuum)))
ok("only nu_L nu_L (and its conjugate) survive, with coefficient v^2/2 -- a Majorana mass for nu_L alone")

<IPython.core.display.Math object>

✓ only nu_L nu_L (and its conjugate) survive, with coefficient v^2/2 -- a Majorana mass for nu_L alone


Every term with a charged lepton or a Goldstone vanishes. What survives is a **Majorana mass term
for $\nu_L$ alone, proportional to $v^2/2$** *(computed)*, plus its conjugate.

### 2.2 What scale?

After electroweak symmetry breaking, the Weinberg operator therefore gives

$$m_\nu \sim \frac{C\, v^2}{\Lambda} ,$$

with $C$ a dimensionless Wilson coefficient.

> **Before running the next cell.** Take $m_\nu \simeq \sqrt{\Delta m^2_{31}}$ from the data (the
> heaviest light neutrino has $m_3 \ge \sqrt{\Delta m^2_{31}}$, with equality when the lightest is
> massless) and set $C = 1$. Is $\Lambda$ nearer the TeV scale or the GUT scale?

In [12]:
# ---- MOVE 2: the scale the operator needs, for C = 1. ----
from anomalies.neutrino_mass.solutions.step_2 import lambda_estimate

m_nu_atm = m_atm_eV / GEV_TO_EV  # GeV
print(f"  step-0 illustrative  m_nu ~ {answer_0:.3g} GeV -> Lambda ~ {lambda_estimate(answer_0, cd._P0_V):.3g} GeV")
Lambda_estimate = lambda_estimate(m_nu_atm, cd._P0_V)
print(f"  sqrt(Delta m^2_31)   m_nu ~ {m_nu_atm:.3g} GeV -> Lambda ~ {Lambda_estimate:.3g} GeV   (computed)")

  step-0 illustrative  m_nu ~ 6.05e-10 GeV -> Lambda ~ 1e+14 GeV
  sqrt(Delta m^2_31)   m_nu ~ 5.01e-11 GeV -> Lambda ~ 1.21e+15 GeV   (computed)


$\Lambda \sim 10^{15}$ GeV *(computed)*, **if** $C = 1$.

**The judgment to keep:** nothing forces $C$ to be $O(1)$. The data fix only the ratio
$C/\Lambda$. Keep this in mind; section 4 shows what $C$ and $\Lambda$ turn out to be in an
explicit model.

---
## 3. Three ways to open the operator

An effective operator is a promise that something heavier was integrated out. At tree level,
$(LH)(LH)$ must come from exchanging one heavy particle between two of its four fields.

> **Before reading the table.** List the ways to split $L, H, L, H$ into two pairs, and for each
> pair say what $SU(2)_L \times U(1)_Y$ quantum numbers the exchanged particle must carry.

There are exactly three tree-level completions *(cited: de Blas, Criado, Pérez-Victoria,
Santiago, arXiv:1711.10391)*:

| completion | pairs | heavy field | spin | SU(2)$_L$ | $Y$ | couples to |
|---|---|---|---|---|---|---|
| type I | $(LH)(LH)$, singlet | $N$ ($\nu_R$) | ½ | 1 | 0 | $\bar L \tilde H N$ |
| type II | $(LL)(HH)$ | $\Delta$ | 0 | 3 | 1 | $L L \Delta$, $H H \Delta^\dagger$ |
| type III | $(LH)(LH)$, triplet | $\Sigma$ | ½ | 3 | 0 | $\bar L \tilde H \Sigma$ |

The minimality rule is: fewer fields, smaller representations, fewer parameters, no ad hoc
symmetries. By that rule, the gauge-singlet $\nu_R$ (type I) wins on every count. This repo never
declares that Lagrangian itself. It imports the already-verified model from `feynlag-models` by
`model_id`:

In [13]:
# ---- MOVE 1: set up.  The type-I model, from the registry, and its new terms. ----
from feynlag_models.registry import build, metadata

answer_3 = "seesaw_type1"
assert cd.check_3(answer_3)

print("feynlag-models maturity:", metadata(answer_3)["maturity_level"])
bundle = build(answer_3)
display(Math(r"\mathcal{L}_{\rm Yuk} = " + latex(bundle.extra["LYukD"])))
display(Math(r"\mathcal{L}_{\rm Maj} = " + latex(bundle.extra["LMaj"])))

[step_3] correct -- got 'seesaw_type1'
feynlag-models maturity: 2


<IPython.core.display.Math object>

<IPython.core.display.Math object>

**Read it.** The Dirac Yukawa $y_\nu$ ties $\nu_R$ to $\nu_L$ through the Higgs, exactly like the
charged-lepton Yukawa of section 1. The Majorana mass $M_R\, \nu_R^T \mathcal{C} \nu_R$ is allowed
**only** because $\nu_R$ is a complete gauge singlet: the ledger of section 1 gives it
$\sum Y = 0$ automatically. This is where lepton number is broken.

---
## 4. The seesaw *is* the Weinberg operator

> **Before running the next cells.** With exactly one $\nu_R$ (one generation), how many physical
> Majorana mass eigenstates should the model have? Roughly how heavy is each one if
> $m_D \ll M_R$? (Hint: think about the eigenvalues of
> $\begin{pmatrix}0 & m_D \\ m_D & M_R\end{pmatrix}$.)

In [14]:
# ---- MOVE 1: set up.  How many mass eigenstates does the bundle carry? ----
answer_4 = len(bundle.extra["masses"])
check_4 = cd.make_check_4(bundle)
assert check_4(answer_4)

[step_4] correct -- got 2


In [15]:
# ---- MOVE 2 + 4: the mass matrix, then the seesaw formula checked against exact Takagi. ----
trace("  the neutral mass matrix:", sp.Eq(sp.Symbol(r"\mathcal{M}_\nu"), bundle.extra["Mnu"], evaluate=False))
trace("  its seesaw approximation for the light state:",
      sp.Eq(sp.Symbol(r"m_\nu^{\rm light}"), bundle.extra["m_light_approx"], evaluate=False))

values = bundle.values()
bench = bundle.benchmark
light_mass = values[bundle.extra["MN1"].s]
heavy_mass = values[bundle.extra["MN2"].s]
light_approx = float(bundle.extra["m_light_approx"].subs(values))
m_D = bench["yv"] * bench["v"] / math.sqrt(2)
rel_diff = abs(abs(light_approx) - light_mass) / light_mass
print(f"\n  benchmark: y_nu = {bench['yv']:.0e}, M_R = {bench['MR']:.0f} GeV")
print(f"  light mass, exact (Takagi)    {light_mass * GEV_TO_EV:.6g} eV   (computed)")
print(f"  light mass, seesaw formula    {light_approx * GEV_TO_EV:.6g} eV   (computed)")
print(f"  relative difference of |m|    {rel_diff:.1e}")
print(f"  (m_D / M_R)^2                 {(m_D / bench['MR']) ** 2:.1e}")
print(f"  heavy mass                    {heavy_mass:.6g} GeV")
assert 0.1 < rel_diff / (m_D / bench["MR"]) ** 2 < 10
ok("exact and seesaw light masses differ by (m_D/M_R)^2 -- the first term the expansion drops")

  the neutral mass matrix:


                  ⎡         √2⋅v⋅yv⎤
                  ⎢   0     ───────⎥
                  ⎢            2   ⎥
\mathcal{M}_\nu = ⎢                ⎥
                  ⎢√2⋅v⋅yv         ⎥
                  ⎢───────    MR   ⎥
                  ⎣   2            ⎦

  its seesaw approximation for the light state:


                       2   2 
                     -v ⋅yv  
m_\nu__{\rm light} = ────────
                       2⋅MR  


  benchmark: y_nu = 1e-06, M_R = 1000 GeV
  light mass, exact (Takagi)    0.0303118 eV   (computed)
  light mass, seesaw formula    -0.0303118 eV   (computed)
  relative difference of |m|    3.1e-14
  (m_D / M_R)^2                 3.0e-14
  heavy mass                    1000 GeV
✓ exact and seesaw light masses differ by (m_D/M_R)^2 -- the first term the expansion drops


**Compare.** The exact diagonalisation and the seesaw formula differ by a few parts in $10^{14}$.
That is not rounding noise: it is the size of $(m_D/M_R)^2$, the first correction the seesaw
expansion drops *(computed)*. The light mass is $m_D^2/M_R$: the heavier $\nu_R$ is, the lighter
$\nu_L$ becomes, hence *seesaw*.

**Read it.** The **minus sign** in $-v^2 y_\nu^2/(2M_R)$ is not a negative mass. For a Majorana
fermion the sign is a phase that a field redefinition ($\nu \to i\nu$) removes. The physical mass
is $|m|$, which is what Takagi factorisation returns.

### 4.1 Closing the loop with section 2

The light mass $y_\nu^2 v^2/(2M_R)$ has exactly the form the Weinberg operator produced in the
vacuum ($\propto v^2/2$), with $C/\Lambda = y_\nu^2/M_R$. Integrating out $\nu_R$ *is* the
Weinberg operator.

> **Before running the next cell.** The benchmark's $\nu_R$ weighs 1 TeV. Section 2 asked for
> $\Lambda \sim 10^{15}$ GeV. Is this benchmark inconsistent with section 2?

In [16]:
# ---- MOVE 4: check.  The effective scale of the benchmark vs section 2's Lambda. ----
Lambda_eff = bench["MR"] / bench["yv"] ** 2
print(f"  type-I benchmark   M_R / y_nu^2 = {Lambda_eff:.3g} GeV   (computed)")
print(f"  section 2, C = 1   Lambda       = {Lambda_estimate:.3g} GeV   (computed)")
assert 0.1 < Lambda_eff / Lambda_estimate < 10
ok("a 1 TeV nu_R with y_nu ~ 1e-6 IS a 1e15 GeV Weinberg scale")

  type-I benchmark   M_R / y_nu^2 = 1e+15 GeV   (computed)
  section 2, C = 1   Lambda       = 1.21e+15 GeV   (computed)
✓ a 1 TeV nu_R with y_nu ~ 1e-6 IS a 1e15 GeV Weinberg scale


Both are around $10^{15}$ GeV *(computed)*, yet the $\nu_R$ weighs only 1 TeV.

**The judgment to keep:** the "$\Lambda \sim 10^{15}$ GeV" of section 2 is the **combination**
$M_R/y_\nu^2$. It can be a GUT-scale $\nu_R$ with $y_\nu \sim 1$, or a TeV $\nu_R$ with
$y_\nu \sim 10^{-6}$. Neutrino masses alone cannot tell these apart.

---
## 5. One $\nu_R$ is not enough

Section 4 had one generation. The data have three flavours and two splittings. With three lepton
generations and one $\nu_R$, the light-neutrino matrix is $m_\nu = -m_D M_R^{-1} m_D^T$, with
$m_D$ a $3\times1$ column.

> **Before running the next cells.** What is the rank of this $3\times3$ matrix? How many light
> neutrinos are massive, and how many *independent* $\Delta m^2$ can it produce? What changes
> with two $\nu_R$?

In [17]:
# ---- peek: the light matrix for one nu_R, fully symbolic. ----
# feynlag's own seesaw formula on a generic m_D (3x1) and M_R (1x1) -- plain
# linear algebra, not a model.
from anomalies.neutrino_mass.solutions.step_5 import light_mass_matrix, light_rank

light_mass_matrix(1)

⎡        2                                    ⎤
⎢   -mDₑ₁        -mDₑ₁⋅mDₘᵤ₁    -mDₑ₁⋅mDₜₐᵤ₁  ⎥
⎢   ───────      ────────────   ───────────── ⎥
⎢     MR₁            MR₁             MR₁      ⎥
⎢                                             ⎥
⎢                        2                    ⎥
⎢-mDₑ₁⋅mDₘᵤ₁       -mDₘᵤ₁       -mDₘᵤ₁⋅mDₜₐᵤ₁ ⎥
⎢────────────      ────────     ──────────────⎥
⎢    MR₁             MR₁             MR₁      ⎥
⎢                                             ⎥
⎢                                        2    ⎥
⎢-mDₑ₁⋅mDₜₐᵤ₁   -mDₘᵤ₁⋅mDₜₐᵤ₁     -mDₜₐᵤ₁     ⎥
⎢─────────────  ──────────────    ─────────   ⎥
⎣     MR₁            MR₁             MR₁      ⎦

Every entry is $-m^D_{\alpha 1} m^D_{\beta 1}/M_1$. Every row is a multiple of the same row: the
matrix is an **outer product** of one column with itself.

In [18]:
# ---- MOVE 3 + 4: recognise the rank, and count what it leaves massless. ----
answer_5 = light_rank(1)
assert cd.check_5(answer_5)
for n in (1, 2):
    r = light_rank(n)
    print(f"  {n} nu_R: rank {r} -> {r} massive, {3 - r} massless light neutrino(s), "
          f"{min(r, 2)} nonzero independent Delta m^2   (computed)")
assert light_rank(1) == 1 and light_rank(2) == 2
ok("one nu_R gives rank 1 (one splitting); two measured splittings need rank >= 2, i.e. two nu_R")

[step_5] correct -- got 1
  1 nu_R: rank 1 -> 1 massive, 2 massless light neutrino(s), 1 nonzero independent Delta m^2   (computed)


  2 nu_R: rank 2 -> 2 massive, 1 massless light neutrino(s), 2 nonzero independent Delta m^2   (computed)


✓ one nu_R gives rank 1 (one splitting); two measured splittings need rank >= 2, i.e. two nu_R


**Recognise.** With one $\nu_R$ there is one massive light neutrino and two massless ones, and
therefore only **one** nonzero $\Delta m^2$ *(computed)*. The data show two, so **the
single-$\nu_R$ seesaw is ruled out by oscillations alone**, before any fit. Two $\nu_R$ give rank 2
*(computed)*: two massive light neutrinos and one *exactly* massless.

**The judgment to keep:** the number of $\nu_R$ is bounded from below by linear algebra, not by
taste. That model was requested rather than built here, because this repo never declares a
Lagrangian. See [`model_requests/seesaw_type1_nN.md`](../../model_requests/seesaw_type1_nN.md).

---
## 6. Two $\nu_R$ against the data

The model the request asked for now exists in `feynlag-models` as `seesaw_type1_2n` (three lepton
generations, two $\nu_R$, maturity L2). We fit its six Dirac Yukawas $y^\nu_{\alpha k}$ to five
measured observables in normal ordering: $\Delta m^2_{21}$, $\Delta m^2_{31}$, $\theta_{12}$,
$\theta_{13}$ and $\theta_{23}$. The heavy masses $M_1 = 1$ TeV and $M_2 = 3$ TeV are held at the
model benchmark. $\delta_{CP}$ is recorded but not fitted, because the model's Yukawas are real,
so it cannot produce CP violation.

**Why a good fit is guaranteed.** The Casas–Ibarra parametrisation *(cited: Ibarra & Ross,
arXiv:hep-ph/0312138v2, Eq. (6), the regression reference for this fit)* runs the seesaw
backwards. It writes $m_D$ in terms of the light masses, the PMNS matrix, the heavy masses, and a
$3\times2$ orthogonal matrix $R$: $m_D \sim U \sqrt{\hat m}\, R\, \sqrt{\hat M}$ (schematically).
For real Yukawas and fixed $M_{1,2}$, count the parameters: two light masses ($m_2, m_3$), three
angles, and **one free angle $z$ in $R$**. That is six, one more than the five observables.

Each trial point substitutes the Yukawas into the model's symbolic $5\times5$ mass matrix and
diagonalises it with feynlag's numeric Takagi factorisation. There is no model rebuild per point.

### 6.1 The fit

> **Before running the next cell.** Six parameters against five observables leaves $-1$ degrees
> of freedom. What will $\chi^2_{\min}$ be? What does that value tell you, and what does it *not*
> tell you?

In [19]:
# ---- MOVE 1 + 2: set up the model, then collect the fit. ----
from anomalies.neutrino_mass.solutions import step_6

model_id = step_6.MODEL_ID
print("feynlag-models maturity:", metadata(model_id)["maturity_level"])
bundle_2n = build(model_id)
trace("  the 5x5 neutral mass matrix the fit diagonalises:",
      sp.Eq(sp.Symbol(r"\mathcal{M}_\nu"), bundle_2n.extra["Mnu"], evaluate=False))
observed = step_6.observed_from_anomaly(anomaly)

fit = step_6.run_fit(bundle_2n, observed)
rows = ["| observable | observed | fitted | pull |", "|---|---|---|---|"]
for name, (value, sigma) in observed.items():
    rows.append(f"| {name} | {value:.5g} ± {sigma:.2g} | {fit['predicted'][name]:.5g} | {fit['pulls'][name]:+.1e} |")
display(Markdown("\n".join(rows)))
print(f"  chi2_min = {fit['chi2']:.2e}, ndof = {fit['ndof']}, converged: {fit['success']}, "
      f"perturbative: {fit['perturbative']}   (computed)")
assert fit["success"] and fit["perturbative"]
ok("the fit converged, with perturbative Yukawas")

feynlag-models maturity: 2


  the 5x5 neutral mass matrix the fit diagonalises:


                  ⎡                                     √2⋅v⋅yvₑ₁    √2⋅v⋅yvₑ₂ ⎤
                  ⎢    0          0            0        ─────────    ───────── ⎥
                  ⎢                                         2            2     ⎥
                  ⎢                                                            ⎥
                  ⎢                                    √2⋅v⋅yvₘᵤ₁   √2⋅v⋅yvₘᵤ₂ ⎥
                  ⎢    0          0            0       ──────────   ────────── ⎥
                  ⎢                                        2            2      ⎥
                  ⎢                                                            ⎥
                  ⎢                                    √2⋅v⋅yvₜₐᵤ₁  √2⋅v⋅yvₜₐᵤ₂⎥
\mathcal{M}_\nu = ⎢    0          0            0       ───────────  ───────────⎥
                  ⎢                                         2            2     ⎥
                  ⎢                                                            ⎥
                  ⎢√2⋅v⋅yvₑ₁

| observable | observed | fitted | pull |
|---|---|---|---|
| Delta m^2_21 (solar) | 7.49e-05 ± 1.9e-06 | 7.49e-05 | -2.9e-09 |
| Delta m^2_31 (atmospheric, normal ordering) | 0.002513 ± 2e-05 | 0.002513 | -2.1e-10 |
| theta_12 | 33.68 ± 0.71 | 33.68 | +6.4e-09 |
| theta_13 | 8.56 ± 0.11 | 8.56 | -6.8e-09 |
| theta_23 | 43.3 ± 0.9 | 43.3 | -1.6e-07 |

  chi2_min = 2.65e-14, ndof = -1, converged: True, perturbative: True   (computed)
✓ the fit converged, with perturbative Yukawas


### 6.2 What $\chi^2 \approx 0$ does not mean

A $\chi^2_{\min}$ of essentially zero with negative degrees of freedom means the model **can**
accommodate the data. It is not evidence that the data **prefer** it. The model's genuine
prediction is structural: rank 2 (section 5) makes the lightest neutrino exactly massless, so
$\sum m_\nu = \sqrt{\Delta m^2_{21}} + \sqrt{\Delta m^2_{31}}$ is fixed by the splittings alone.

In [20]:
# ---- MOVE 3 + 4: recognise the structural prediction, and the SM contrast. ----
m1, m2, m3 = fit["masses_eV"]
print(f"  light masses [eV]: m1 = {m1}, m2 = {m2:.4g}, m3 = {m3:.4g}   (computed)")
print(f"  sum m_nu = {fit['sum_m_nu_eV']:.4g} eV   (computed; lightest state massless)")
print(f"  chi2 over the two splittings: SM = {fit['chi2_sm_dm2']:.3g}, model = {fit['chi2_model_dm2']:.1e}")

answer_6 = fit["predicted"][step_6.FIT_OBSERVABLES[1]] / fit["predicted"][step_6.FIT_OBSERVABLES[0]]
assert cd.make_check_6(anomaly)(answer_6)
assert m1 == 0
ok("m1 = 0 exactly, as the rank-2 argument of section 5 said it must be")

  light masses [eV]: m1 = 0.0, m2 = 0.008654, m3 = 0.05013   (computed)
  sum m_nu = 0.05878 eV   (computed; lightest state massless)
  chi2 over the two splittings: SM = 1.73e+04, model = 8.7e-18
[step_6] correct -- got 33.5514
✓ m1 = 0 exactly, as the rank-2 argument of section 5 said it must be


**Read it.** The checkpoint asks for $\Delta m^2_{31}/\Delta m^2_{21} \approx 33.5$. This ratio
is the mass **hierarchy**: the atmospheric splitting is about 30 times the solar one. A rank-1
light sector has no second splitting, so it cannot produce any ratio at all.

### 6.3 The Yukawas are not unique

> **Before running the next cell.** The spare Casas–Ibarra angle $z$ is a flat direction of the
> $\chi^2$. If you start the fit from a different point (the benchmark with some signs flipped),
> which quantities should change, and which cannot?

In [21]:
# ---- MOVE 4: check.  A second start: different Yukawas, same spectrum. ----
fit_alt = step_6.run_fit(bundle_2n, observed, x0=step_6.alt_start(bundle_2n))
rows = ["| | " + " | ".join(f"${latex(sp.Symbol(k))}$" for k in fit["yv"]) + " | $\\chi^2$ | $m_2$, $m_3$ [eV] |",
        "|---" * (len(fit["yv"]) + 3) + "|"]
for label, f in (("benchmark start", fit), ("sign-flipped start", fit_alt)):
    rows.append(f"| {label} | " + " | ".join(f"{y:.2e}" for y in f["yv"].values())
                + f" | {f['chi2']:.1e} | {f['masses_eV'][1]:.5g}, {f['masses_eV'][2]:.5g} |")
display(Markdown("\n".join(rows) + "\n\n*(computed)*"))
assert np.allclose(fit_alt["masses_eV"], fit["masses_eV"], rtol=1e-6, atol=1e-12)
assert not np.allclose(list(fit_alt["yv"].values()), list(fit["yv"].values()), rtol=1e-2)
ok("different Yukawas, identical light spectrum -- the data fix m_D M^-1 m_D^T, not m_D")

| | $y^\nu_{e 1}$ | $y^\nu_{e 2}$ | $y^\nu_{\mu 1}$ | $y^\nu_{\mu 2}$ | $y^\nu_{\tau 1}$ | $y^\nu_{\tau 2}$ | $\chi^2$ | $m_2$, $m_3$ [eV] |
|---|---|---|---|---|---|---|---|---|
| benchmark start | 1.59e-07 | 5.40e-07 | 9.05e-07 | -4.50e-07 | 8.92e-07 | 6.39e-07 | 2.7e-14 | 0.0086545, 0.05013 |
| sign-flipped start | 4.42e-08 | -6.01e-07 | 9.39e-07 | -1.09e-07 | -7.13e-07 | 1.13e-06 | 1.8e-14 | 0.0086545, 0.05013 |

*(computed)*

✓ different Yukawas, identical light spectrum -- the data fix m_D M^-1 m_D^T, not m_D


### 6.4 How visible is $\nu_R$?

Pinning down the Yukawas needs processes that probe $\nu_R$ directly. At tree level, $\nu_R$ mixes
with $\nu_L$ with angle $\theta_{\alpha k} \simeq (m_D)_{\alpha k}/M_k = y^\nu_{\alpha k} v/(\sqrt2 M_k)$.

In [22]:
# ---- MOVE 2: the largest active-sterile mixing at the best fit. ----
bench_2n = bundle_2n.benchmark
theta_max = max(abs(y) * bench_2n["v"] / (math.sqrt(2) * bench_2n[f"MR{name[-1]}"])
                for name, y in fit["yv"].items())
print(f"  largest active-sterile mixing |theta| ~ {theta_max:.1e}   (computed)")

  largest active-sterile mixing |theta| ~ 1.6e-07   (computed)


The mixing is of order $10^{-7}$ *(computed)*. A 1–3 TeV $\nu_R$ is a gauge singlet, so it is
produced and decays only through this mixing, with rates suppressed by $|\theta|^2 \sim 10^{-14}$.
With couplings this small, the model explains the masses, but its $\nu_R$ is essentially out of
direct reach.

---
## 7. Recap

**The numbers**, each one produced by a cell above:

In [23]:
summary = [
    ("0", "dimensional estimate $y^2v^2/M$", f"{answer_0 * GEV_TO_EV:.2g} eV vs $\\sqrt{{\\Delta m^2_{{31}}}}$ = {m_atm_eV:.2g} eV", "computed"),
    ("1", "no dim ≤ 4 $\\nu$ mass term in the SM", f"{answer_1} invariant (charged lepton only)", "computed"),
    ("2", "Weinberg operator is the unique dim-5 term; gives a $\\nu_L$ Majorana mass", f"$\\Lambda/C$ ≈ {Lambda_estimate:.1e} GeV", "computed"),
    ("3", "type-I seesaw is the minimal tree-level completion", "`seesaw_type1`", "cited (arXiv:1711.10391)"),
    ("4", "seesaw reproduces Weinberg with $C/\\Lambda = y^2/M_R$", f"$M_R/y^2$ = {Lambda_eff:.1e} GeV", "computed"),
    ("5", "one $\\nu_R$ gives rank 1: only one $\\Delta m^2$", f"rank {light_rank(1)} (1 $\\nu_R$), {light_rank(2)} (2 $\\nu_R$)", "computed"),
    ("6", "two $\\nu_R$ accommodate the data; $m_1 = 0$", f"$\\chi^2_{{\\min}}$ = {fit['chi2']:.0e} (ndof = {fit['ndof']}), $\\sum m_\\nu$ = {fit['sum_m_nu_eV']:.3g} eV", "computed"),
]
rows = ["| § | result | key number | provenance |", "|---|---|---|---|"]
rows += [f"| {s} | {r} | {n} | {p} |" for s, r, n, p in summary]
display(Markdown("\n".join(rows)))

| § | result | key number | provenance |
|---|---|---|---|
| 0 | dimensional estimate $y^2v^2/M$ | 0.61 eV vs $\sqrt{\Delta m^2_{31}}$ = 0.05 eV | computed |
| 1 | no dim ≤ 4 $\nu$ mass term in the SM | 1 invariant (charged lepton only) | computed |
| 2 | Weinberg operator is the unique dim-5 term; gives a $\nu_L$ Majorana mass | $\Lambda/C$ ≈ 1.2e+15 GeV | computed |
| 3 | type-I seesaw is the minimal tree-level completion | `seesaw_type1` | cited (arXiv:1711.10391) |
| 4 | seesaw reproduces Weinberg with $C/\Lambda = y^2/M_R$ | $M_R/y^2$ = 1.0e+15 GeV | computed |
| 5 | one $\nu_R$ gives rank 1: only one $\Delta m^2$ | rank 1 (1 $\nu_R$), 2 (2 $\nu_R$) | computed |
| 6 | two $\nu_R$ accommodate the data; $m_1 = 0$ | $\chi^2_{\min}$ = 3e-14 (ndof = -1), $\sum m_\nu$ = 0.0588 eV | computed |

**The mechanics.**

0. Dimensional analysis already points to a heavy scale: $m_\nu \sim y^2 v^2/M$.
1. The SM's field content, together with the hypercharge ledger, leaves no dimension-$\le 4$
   neutrino mass, and accidental lepton number protects it.
2. The first operator that breaks that protection is the unique dimension-5 Weinberg operator. In
   the vacuum it is a $\nu_L$ Majorana mass.
3. Opening it at tree level gives exactly three options; the singlet $\nu_R$ is minimal.
4. Integrating out $\nu_R$ reproduces the Weinberg operator, with $C/\Lambda = y_\nu^2/M_R$.
5. Two measured splittings need a rank-2 light matrix, so at least two $\nu_R$.
6. Two $\nu_R$ accommodate the data, with the structural prediction $m_1 = 0$.

**The tools, in the order you should reach for them.**

| question | tool |
|---|---|
| is any term allowed at all? | a hypercharge ledger from `field.reps` |
| which invariants exist, exhaustively? | `feynlag.suggest_yukawa(..., max_dim=n)` |
| what does an operator do at low energy? | `expr.subs({G⁺: 0, H⁰: v/√2})`, term by term |
| which model completes it? | the tree-level table, then `feynlag_models.registry.build(model_id)` |
| how many massive states? | `feynlag.seesaw_light_mass` + `Matrix.rank` |
| exact spectrum and mixing? | feynlag's Takagi factorisation (inside the bundle, or `step_6.OscillationPredictor`) |
| does it fit? | `step_6.run_fit` (stage-1 $\chi^2$ via `fit.stage1.minimize_chi2`) |

**The traps**, each of which a cell above turned into a check:

- **$\Lambda$ is not a particle mass.** The data fix $C/\Lambda$; the seesaw's
  $M_R/y_\nu^2 \sim 10^{15}$ GeV is a TeV $\nu_R$ with tiny couplings just as well as a GUT-scale
  one.
- **The Majorana minus sign is a phase**, not a negative mass. Physical masses are Takagi
  singular values.
- **$\chi^2 \approx 0$ with ndof $< 0$ is accommodation, not preference.** The genuine prediction
  is $m_1 = 0$.
- **Oscillations fix $m_D M_R^{-1} m_D^T$, not $m_D$.** Two fits with different Yukawas give the
  same spectrum.
- **Everything here assumes normal ordering**, the NuFIT column that was fitted.

**The punchline.** Two facts from the data, two nonzero splittings, already force a *new field*,
a *broken accidental symmetry*, and *at least two* copies of that field, before any fit is run.
The fit then only confirms that the minimal choice works.

**Where to go next.** The natural next rungs are cosmology's bound on $\sum m_\nu$ and
neutrinoless double-beta decay's bound on the effective Majorana mass. Both need sourced limits
in `anomaly.yaml` (`tensions_with_other_data` is still `TODO_VERIFY`). For the model side,
feynlag's `examples/SM_Seesaw_Tutorial` builds the seesaw Lagrangian and its Feynman rules
directly.